
# Intelligent Course Recommendation System (RAG)
This notebook implements an advanced Retrieval-Augmented Generation (RAG) system for course recommendations. 

**Key Improvements in this version:**
1. **Object-Oriented Design**: Core logic is encapsulated in a `CourseRAG` class.
2. **Advanced Retrieval**: Implements a two-stage retrieval pipeline (Bi-Encoder for fast semantic search + Cross-Encoder for high-precision re-ranking).
3. **Data Cleaning**: Improved encoding handling to fix text corruption (e.g., garbled time formats).
4. **Structured LLM Output**: Uses Groq's JSON mode for reliable, parsable recommendations.
5. **Interactive UI**: Includes `ipywidgets` for a seamless search experience.

In [3]:
import os
import csv
import json
import math
from pathlib import Path
from typing import Any, Dict, List, Tuple

import pandas as pd
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer, CrossEncoder
import ipywidgets as widgets
from IPython.display import display, clear_output
from dotenv import load_dotenv

# Load environment variables (ensure you have a .env file with GROQ_API_KEY)
load_dotenv()

# Optional: Groq LLM integration
USE_GROQ = bool(os.environ.get("GROQ_API_KEY"))
if USE_GROQ:
    from groq import Groq

# Constants
DATA_DIR = Path(".")
COURSES_CSV = DATA_DIR / "courses_full_dataset_combined_courses.csv"
REVIEWS_CSV = DATA_DIR / "course_review.csv"
DB_PATH = DATA_DIR / "chroma_db"

if not COURSES_CSV.exists():
    COURSES_CSV = DATA_DIR / "cmu_labeled_llm_final.csv"

print(f"Courses CSV path: {COURSES_CSV} - Exists: {COURSES_CSV.exists()}")
print(f"Reviews CSV path: {REVIEWS_CSV} - Exists: {REVIEWS_CSV.exists()}")

Courses CSV path: courses_full_dataset_combined_courses.csv - Exists: True
Reviews CSV path: course_review.csv - Exists: True


## 1. Data Cleaning and Preprocessing
Robust data parsers to handle course schedules, clean text encoding issues, and aggregate review data.

In [4]:
def parse_days(weekday_str: str) -> List[str]:
    if not weekday_str or weekday_str == "TBA":
        return []
    valid_chars = {'M', 'T', 'W', 'R', 'F', 'S', 'U'}
    return [char for char in str(weekday_str).upper() if char in valid_chars]

def parse_time_slots(start_str: str, end_str: str) -> List[str]:
    if not start_str or not end_str:
        return []
    
    # Clean up common encoding artifacts (e.g., garbled 'ä¸')
    start_str = start_str.encode('ascii', 'ignore').decode('ascii').strip().upper()
    end_str = end_str.encode('ascii', 'ignore').decode('ascii').strip().upper()

    def to_mins(t_str):
        try:
            parts = t_str.replace(".", "").split()
            if len(parts) != 2: return -1
            time_part, period = parts
            h, m = map(int, time_part.split(":"))
            if period == "PM" and h != 12: h += 12
            if period == "AM" and h == 12: h = 0
            return h * 60 + m
        except:
            return -1

    start_mins, end_mins = to_mins(start_str), to_mins(end_str)
    if start_mins == -1 or end_mins == -1:
        return []

    slots = []
    current_mins = start_mins
    while current_mins < end_mins:
        h, m = current_mins // 60, current_mins % 60
        period = "PM" if h >= 12 else "AM"
        display_h = h - 12 if h > 12 else (12 if h == 0 else h)
        slots.append(f"{display_h}:{m:02d} {period}")
        current_mins += 30
    return slots

def load_and_clean_data(courses_path: Path, reviews_path: Path):
    courses = []
    # Use utf-8 with error replacement to avoid decoding crashes
    with open(courses_path, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        for row in reader:
            weekday, start, end = row.get("weekday", ""), row.get("start", ""), row.get("end", "")
            row["_days"] = parse_days(weekday)
            row["_times"] = parse_time_slots(start, end)
            
            clean_start = start.encode('ascii', 'ignore').decode('ascii').strip()
            clean_end = end.encode('ascii', 'ignore').decode('ascii').strip()
            
            row["_meetingTime"] = f"{weekday} {clean_start}-{clean_end}" if weekday and clean_start else "TBA"
            
            skills_str = str(row.get("skills", "")).replace("[", "").replace("]", "").replace("'", "")
            row["skills"] = [s.strip() for s in skills_str.split(",") if s.strip()]
            courses.append(row)

    print(f"Loaded {len(courses)} courses.")
    return courses

# Load Data
COURSES = load_and_clean_data(COURSES_CSV, REVIEWS_CSV)
COURSES_BY_ID = {str(c.get("course_id", "")): c for c in COURSES}

Loaded 8450 courses.


## 2. The CourseRAG Engine
This class handles vectorization, the two-stage retrieval process (Bi-encoder + Cross-encoder), and structured LLM generation.

In [6]:
class CourseRAG:
    def __init__(self, db_path: str, use_groq: bool = False):
        self.db_path = Path(db_path)
        self.db_path.mkdir(exist_ok=True)
        
        # Initialize ChromaDB
        self.client = chromadb.PersistentClient(
            path=str(self.db_path),
            settings=Settings(anonymized_telemetry=False, allow_reset=True)
        )
        self.collection = self.client.get_or_create_collection(
            name="courses_v2", metadata={"hnsw:space": "cosine"}
        )
        
        # Models
        print("Loading Bi-Encoder...")
        self.bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        print("Loading Cross-Encoder for Re-ranking...")
        self.cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        
        # LLM
        self.use_groq = use_groq
        if self.use_groq:
            self.llm_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
            self.llm_model = "llama-3.3-70b-versatile"

    def _create_document(self, course: Dict) -> str:
        parts = []
        if course.get("course_id"): parts.append(f"Course: {course['course_id']}")
        if course.get("course_name"): parts.append(f"Title: {course['course_name']}")
        desc = course.get("description_clean") or course.get("description", "")
        if desc: parts.append(f"Description: {desc}")
        if course.get("industry"): parts.append(f"Industry: {course['industry']}")
        if course.get("skills"): parts.append(f"Skills: {', '.join(course['skills'])}")
        return " | ".join(parts)

    def ingest_data(self, courses: List[Dict]):
        if self.collection.count() > 0:
            print(f"Collection already populated with {self.collection.count()} items. Skipping ingest.")
            return

        documents, metadatas, ids = [], [], []
        seen_ids = set()

        for idx, row in enumerate(courses):
            # 获取 ID，如果没有 course_id 则用索引生成一个
            course_id = str(row.get("course_id", f"idx_{idx}"))
            
            if course_id in seen_ids:
                continue
                
            seen_ids.add(course_id)
            documents.append(self._create_document(row))
            ids.append(course_id)
            metadatas.append({
                "course_name": row.get("course_name", ""),
                "industry": row.get("industry", ""),
            })

        print(f"Embedding {len(documents)} unique documents...")
        embeddings = self.bi_encoder.encode(documents, show_progress_bar=True).tolist()

        batch_size = 5000
        for b in range(math.ceil(len(ids) / batch_size)):
            s, e = b * batch_size, (b + 1) * batch_size
            self.collection.upsert(
                ids=ids[s:e], 
                embeddings=embeddings[s:e], 
                documents=documents[s:e], 
                metadatas=metadatas[s:e]
            )
        print("Ingestion complete.")

    def search(self, query: str, top_k: int = 10, schedule_filter: Dict = None) -> List[Tuple[float, Dict]]:
        # Stage 1: Fast Vector Search
        initial_k = top_k * 5 # Fetch more for re-ranking
        results = self.collection.query(query_texts=[query], n_results=initial_k)
        
        candidates = []
        for i, cid in enumerate(results["ids"][0]):
            course = COURSES_BY_ID.get(cid)
            if not course: continue
            
            # Apply hard schedule constraints
            if schedule_filter:
                valid = True
                for d in course.get("_days", []):
                    if d not in schedule_filter or any(t not in schedule_filter[d] for t in course.get("_times", [])):
                        valid = False
                        break
                if not valid: continue
                
            candidates.append((results["documents"][0][i], course))

        if not candidates:
            return []

        # Stage 2: Cross-Encoder Re-ranking
        pairs = [[query, doc] for doc, _ in candidates]
        cross_scores = self.cross_encoder.predict(pairs)
        
        scored_results = []
        for i, score in enumerate(cross_scores):
            scored_results.append((float(score), candidates[i][1]))
            
        # Sort by re-ranked score descending
        scored_results.sort(key=lambda x: x[0], reverse=True)
        return scored_results[:top_k]

    def generate_json_advice(self, query: str, top_k: int = 5) -> Dict:
        if not self.use_groq:
            return {"error": "Groq API not configured."}

        hits = self.search(query, top_k=top_k)
        if not hits:
            return {"advice": "No matching courses found."}

        context_blocks = []
        for score, c in hits:
            context_blocks.append(
                f"ID: {c.get('course_id')} | Title: {c.get('course_name')} | Desc: {c.get('description_clean')[:300]}"
            )
        context_str = "\n".join(context_blocks)

        prompt = f"""You are an academic advisor. Based strictly on the context, answer the query.
Output your response entirely in valid JSON format containing two keys:
1. "recommended_courses": A list of course IDs (e.g., ["15-112", "10-601"]).
2. "explanation": A concise explanation of why these courses fit the user's goal.

Query: {query}
Context:
{context_str}
"""
        response = self.llm_client.chat.completions.create(
            model=self.llm_model,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0.2,
            max_tokens=500
        )
        return json.loads(response.choices[0].message.content)

# Initialize System
rag_system = CourseRAG(db_path=str(DB_PATH), use_groq=USE_GROQ)
rag_system.ingest_data(COURSES)

Loading Bi-Encoder...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4471.82it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading Cross-Encoder for Re-ranking...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4115.44it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 3145 unique documents...


Batches: 100%|██████████| 99/99 [01:05<00:00,  1.52it/s]


Ingestion complete.


## 3. Interactive Search UI
Use the widgets below to test the RAG system directly within the notebook.

In [7]:
# Create UI Elements
query_input = widgets.Text(
    value='I want to learn about product management and prototyping',
    placeholder='Type your learning goals...',
    description='Goal:',
    layout=widgets.Layout(width='80%')
)

search_btn = widgets.Button(
    description='Search & Analyze',
    button_style='primary',
    icon='search'
)

output_area = widgets.Output()

def on_search_clicked(b):
    with output_area:
        clear_output()
        print("🔍 Searching and re-ranking courses...")
        query = query_input.value
        
        # Display raw retrieval results
        hits = rag_system.search(query, top_k=5)
        print("\n--- Top Retrieved Courses (Cross-Encoder Re-ranked) ---")
        for score, course in hits:
            cid = course.get("course_id", "N/A")
            name = course.get("course_name", "Unknown")
            time = course.get("_meetingTime", "TBA")
            print(f"[{score:.2f}] {cid} - {name} ({time})")
            
        # Display LLM Generation (if configured)
        if USE_GROQ:
            print("\n🤖 Generating AI Advice (JSON Mode)...")
            try:
                advice_json = rag_system.generate_json_advice(query, top_k=5)
                print(json.dumps(advice_json, indent=2))
            except Exception as e:
                print(f"Error generating advice: {e}")

search_btn.on_click(on_search_clicked)

# Display UI
display(widgets.VBox([
    widgets.HTML("<h3>Find Your Courses</h3>"),
    query_input, 
    search_btn, 
    output_area
]))